# `cp_parallel` tutorial

`cp_parallel` runs [CellProfiler](https://cellprofiler.org/) headless on one or many plates (or wells) in
parallel, and writes a log file per run. This notebook walks through every mode the package supports:

1. A single plate
2. Multiple plates in parallel
3. Well-level grouping
4. A LoadData CSV instead of a raw image directory
5. Batch processing by image-set range
6. Running inside an Apptainer/Singularity container (HPC)

See the [README](../README.md) for installation instructions and the full API reference.

> **About this notebook:** to make it runnable anywhere (including CI, with no CellProfiler install),
> the cells below replace the real `cellprofiler` executable with a tiny stand-in that mimics its
> behavior (writes some text to stderr and exits 0). **This substitution is only for the tutorial** —
> see the [final section](#Using-this-with-a-real-CellProfiler-install) for what changes when you point
> `cp_parallel` at your own pipelines.

In [1]:
import pathlib
import shutil
import tempfile
from contextlib import contextmanager
from unittest.mock import patch

import cp_parallel

print(f"cp_parallel loaded from: {cp_parallel.__file__}")

cp_parallel loaded from: /home/jenna/CP_Parallel/src/cp_parallel/__init__.py


## Demo workspace and a stand-in for `cellprofiler`

The helpers below create a scratch directory with a fake pipeline file, a fake image directory, and a
fake LoadData CSV -- just enough for `cp_parallel` to pass its path checks -- and a `simulate_cellprofiler`
context manager that replaces the real subprocess call. **In your own notebooks, skip this cell**: as
long as `cellprofiler` is on your `PATH` (e.g. you're in the right conda environment), `cp_parallel` calls
the real thing.

In [2]:
workspace = pathlib.Path(tempfile.mkdtemp(prefix="cp_parallel_tutorial_"))

pipeline_path = workspace / "analysis.cppipe"
pipeline_path.write_text("CellProfiler Pipeline: http://www.cellprofiler.org\n")

loaddata_path = workspace / "load_data.csv"
loaddata_path.write_text("Metadata_Plate,Metadata_Well\nPlate1,A01\n")


def make_images_dir(name: str) -> pathlib.Path:
    images_dir = workspace / name
    images_dir.mkdir(exist_ok=True)
    (images_dir / "well_A01_z01.tif").write_bytes(b"")
    return images_dir


@contextmanager
def simulate_cellprofiler():
    """Stand in for the real `cellprofiler` binary so this notebook runs without it installed.

    Also swaps `ProcessPoolExecutor` for `ThreadPoolExecutor`: a real process pool would need to
    pickle the mocked `subprocess.run` to send it to a worker process, which doesn't work. Your own
    (un-mocked) code doesn't need this -- real subprocess calls pickle and run across processes fine.
    """
    import subprocess
    from concurrent.futures import ThreadPoolExecutor
    from contextlib import ExitStack

    def fake_run(args, capture_output=True):
        message = f"CellProfiler (simulated) ran: {' '.join(str(a) for a in args)}\n"
        return subprocess.CompletedProcess(
            args=args, returncode=0, stdout=b"", stderr=message.encode()
        )

    # contextlib.ExitStack (not a parenthesized `with`) keeps this compatible with
    # Python 3.8, which is what CellProfiler conda environments commonly run.
    with ExitStack() as stack:
        stack.enter_context(
            patch("cp_parallel.cp_parallel.subprocess.run", side_effect=fake_run)
        )
        stack.enter_context(
            patch(
                "cp_parallel.cp_parallel.shutil.which",
                return_value="/usr/bin/apptainer",
            )
        )
        stack.enter_context(
            patch("cp_parallel.cp_parallel.ProcessPoolExecutor", ThreadPoolExecutor)
        )
        yield


print(f"Workspace: {workspace}")

Workspace: /tmp/cp_parallel_tutorial_w1x_ginc


## 1. A single plate

The same function handles one plate or many -- parallelism is just "how many entries did you pass in."
Each entry needs a pipeline, an output directory, and either `path_to_images` or `path_to_loaddata`.

In [3]:
plate_info_dictionary = {
    "Plate1": {
        "path_to_pipeline": pipeline_path,
        "path_to_images": make_images_dir("Plate1_images"),
        "path_to_output": workspace / "outputs" / "Plate1",
    },
}

log_dir = workspace / "logs" / "single_plate"

with simulate_cellprofiler():
    results = cp_parallel.run_cellprofiler_parallel(
        plate_info_dictionary=plate_info_dictionary,
        run_name="whole_image_qc",
        log_dir=log_dir,
    )

print(f"Return code: {results[0].returncode}")
print((log_dir / "Plate1_whole_image_qc_run.log").read_text())

Return code: 0
CellProfiler (simulated) ran: cellprofiler -c -r -p /tmp/cp_parallel_tutorial_w1x_ginc/analysis.cppipe -i /tmp/cp_parallel_tutorial_w1x_ginc/Plate1_images -o /tmp/cp_parallel_tutorial_w1x_ginc/outputs/Plate1



## 2. Multiple plates in parallel

Add more entries to the dictionary and they run concurrently (up to the number of CPUs available).

In [4]:
plate_info_dictionary = {
    f"Plate{i}": {
        "path_to_pipeline": pipeline_path,
        "path_to_images": make_images_dir(f"Plate{i}_images"),
        "path_to_output": workspace / "outputs" / f"Plate{i}",
    }
    for i in range(1, 4)
}

log_dir = workspace / "logs" / "multi_plate"

with simulate_cellprofiler():
    results = cp_parallel.run_cellprofiler_parallel(
        plate_info_dictionary=plate_info_dictionary,
        run_name="feature_extraction",
        log_dir=log_dir,
    )

print(f"Ran {len(results)} plates; log files:")
for log_file in sorted(log_dir.glob("*.log")):
    print(" -", log_file.name)

Ran 3 plates; log files:
 - Plate1_feature_extraction_run.log
 - Plate2_feature_extraction_run.log
 - Plate3_feature_extraction_run.log


## 3. Well-level grouping

Set `group_level="well"`. Each entry's key becomes the `-g Well=...` value passed to CellProfiler, and a
`plate_name` is required on each entry so log files can be grouped per plate.

In [5]:
images_dir = make_images_dir("Plate1_wells")

plate_info_dictionary = {
    "A01": {
        "plate_name": "Plate1",
        "path_to_pipeline": pipeline_path,
        "path_to_images": images_dir,
        "path_to_output": workspace / "outputs" / "Plate1",
    },
    "A02": {
        "plate_name": "Plate1",
        "path_to_pipeline": pipeline_path,
        "path_to_images": images_dir,
        "path_to_output": workspace / "outputs" / "Plate1",
    },
}

log_dir = workspace / "logs" / "well_level"

with simulate_cellprofiler():
    cp_parallel.run_cellprofiler_parallel(
        plate_info_dictionary=plate_info_dictionary,
        run_name="well_qc",
        group_level="well",
        log_dir=log_dir,
    )

print("Log files are nested per plate:")
for log_file in sorted(log_dir.rglob("*.log")):
    print(" -", log_file.relative_to(log_dir))

Log files are nested per plate:
 - Plate1/A01_well_qc_run.log
 - Plate1/A02_well_qc_run.log


## 4. A LoadData CSV instead of a raw image directory

Swap `path_to_images` for `path_to_loaddata` and CellProfiler is invoked with `--data-file` instead of
`-i`.

In [6]:
plate_info_dictionary = {
    "Plate1": {
        "path_to_pipeline": pipeline_path,
        "path_to_loaddata": loaddata_path,
        "path_to_output": workspace / "outputs" / "Plate1_loaddata",
    },
}

log_dir = workspace / "logs" / "loaddata"

with simulate_cellprofiler():
    cp_parallel.run_cellprofiler_parallel(
        plate_info_dictionary=plate_info_dictionary,
        run_name="loaddata_run",
        log_dir=log_dir,
    )

print((log_dir / "Plate1_loaddata_run_run.log").read_text())

CellProfiler (simulated) ran: cellprofiler -c -r -p /tmp/cp_parallel_tutorial_w1x_ginc/analysis.cppipe --data-file /tmp/cp_parallel_tutorial_w1x_ginc/load_data.csv -o /tmp/cp_parallel_tutorial_w1x_ginc/outputs/Plate1_loaddata



## 5. Batch processing by image-set range

For a large run (e.g. a timelapse dataset, or any LoadData CSV with many image sets), split the image sets into batches and call `run_cellprofiler_parallel` once per batch, using `first_image_set`/`last_image_set` (`-f`/`-l`) to select each range. `max_workers` caps concurrency independent of how many batches you're running.

In [7]:
log_dir = workspace / "logs" / "batch_run"

for batch_index, (first, last) in enumerate([(1, 100), (101, 200)]):
    plate_info_dictionary = {
        f"batch_{batch_index}": {
            "path_to_pipeline": pipeline_path,
            "path_to_loaddata": loaddata_path,
            "path_to_output": workspace / "outputs" / "batch_run",
            "first_image_set": first,
            "last_image_set": last,
        }
    }
    with simulate_cellprofiler():
        cp_parallel.run_cellprofiler_parallel(
            plate_info_dictionary=plate_info_dictionary,
            run_name="batch_analysis",
            max_workers=1,
            log_dir=log_dir,
        )

for log_file in sorted(log_dir.glob("*.log")):
    print(f"--- {log_file.name} ---")
    print(log_file.read_text())

--- batch_0_batch_analysis_run.log ---
CellProfiler (simulated) ran: cellprofiler -c -r -p /tmp/cp_parallel_tutorial_w1x_ginc/analysis.cppipe --data-file /tmp/cp_parallel_tutorial_w1x_ginc/load_data.csv -o /tmp/cp_parallel_tutorial_w1x_ginc/outputs/batch_run -f 1 -l 100

--- batch_1_batch_analysis_run.log ---
CellProfiler (simulated) ran: cellprofiler -c -r -p /tmp/cp_parallel_tutorial_w1x_ginc/analysis.cppipe --data-file /tmp/cp_parallel_tutorial_w1x_ginc/load_data.csv -o /tmp/cp_parallel_tutorial_w1x_ginc/outputs/batch_run -f 101 -l 200



## 6. Running inside an Apptainer/Singularity container (HPC)

Pass `run_with_apptainer` with a path to a `.sif` image and `cp_parallel` wraps each command as
`apptainer exec <image> cellprofiler ...` (or `singularity exec ...`, whichever it finds on `PATH`). On an
HPC job, also set `max_workers` to match your allocation -- `cp_parallel` validates it against the CPUs
available to the *job* (via CPU affinity), not the full node.

This cell's `simulate_cellprofiler()` also pretends `apptainer` is on `PATH` so we can see the generated
command without a real container runtime installed.

In [8]:
plate_info_dictionary = {
    "Plate1": {
        "path_to_pipeline": pipeline_path,
        "path_to_images": make_images_dir("Plate1_hpc"),
        "path_to_output": workspace / "outputs" / "Plate1_hpc",
    },
}

log_dir = workspace / "logs" / "hpc"

with simulate_cellprofiler():
    cp_parallel.run_cellprofiler_parallel(
        plate_info_dictionary=plate_info_dictionary,
        run_name="hpc_run",
        run_with_apptainer="/path/to/cellprofiler.sif",
        max_workers=1,
        log_dir=log_dir,
    )

print((log_dir / "Plate1_hpc_run_run.log").read_text())

CellProfiler (simulated) ran: apptainer exec /path/to/cellprofiler.sif cellprofiler -c -r -p /tmp/cp_parallel_tutorial_w1x_ginc/analysis.cppipe -i /tmp/cp_parallel_tutorial_w1x_ginc/Plate1_hpc -o /tmp/cp_parallel_tutorial_w1x_ginc/outputs/Plate1_hpc



## Using this with a real CellProfiler install

Everything above ran against a simulated `cellprofiler`. In your own analysis, drop the
`simulate_cellprofiler()` context manager entirely and point the dictionary at real files:

```python
import cp_parallel

plate_info_dictionary = {
    "Plate1": {
        "path_to_pipeline": "pipelines/analysis.cppipe",
        "path_to_images": "data/Plate1/images",
        "path_to_output": "outputs/Plate1",
    },
}

cp_parallel.run_cellprofiler_parallel(
    plate_info_dictionary=plate_info_dictionary,
    run_name="whole_image_qc",
)
```

As long as `cellprofiler` (and, for the HPC example, `apptainer`/`singularity`) is on your `PATH`,
`cp_parallel` calls the real thing. See the [README](../README.md) for the full
`plate_info_dictionary` reference and API docs.

In [9]:
# Clean up the scratch workspace created for this tutorial.
shutil.rmtree(workspace)
print(f"Removed {workspace}")

Removed /tmp/cp_parallel_tutorial_w1x_ginc
